# MPS and Pauli-space basics

This notebook covers the basic ITensor-side utilities in EDKit: `vec2mps`, `mps2vec`, `mps2pmps`, `pmps2mpo`, and `mpo2pmpo`.


In [ ]:
using LinearAlgebra
DEV = true

function find_local_edkit(start = pwd())
    dir = abspath(start)
    while true
        candidate = joinpath(dir, "src", "EDKit.jl")
        isfile(candidate) && return candidate
        parent = dirname(dir)
        parent == dir && error("Could not locate src/EDKit.jl from $(start)")
        dir = parent
    end
end

if DEV
    include(find_local_edkit())
    using .EDKit
else
    using EDKit
end
using ITensors
using ITensorMPS


In [ ]:
L = 5
s = siteinds(2, L)
ps = siteinds("Pauli", L)

psi_vec = randn(ComplexF64, 2^L) |> normalize
psi_mps = vec2mps(psi_vec, s)
psi_back = mps2vec(psi_mps)

pmps = mps2pmps(psi_mps, ps)
rho_mpo = pmps2mpo(pmps, s)
pmpo = mpo2pmpo(rho_mpo, ps)

bond_dims = [linkdim(psi_mps, b) for b in 1:length(psi_mps)-1]


In [ ]:
summary = (; 
    system_size = L,
    roundtrip_error = norm(psi_vec - psi_back),
    max_mps_bond_dimension = maximum(bond_dims; init = 1),
    pmps_length = length(pmps),
    mpo_length = length(rho_mpo),
    pmpo_length = length(pmpo),
)

summary
